In [ ]:
# Operator Overloading
    # Operator overloading allows custom classes to seamlessly integrate with Python's core language mechanics.
    # The Mechanism: (e.g., __add__)
    # Interface Emulation: Overloading methods are completely optional.
# Constructors and Expressions: __init__ and __sub__
from number import Number # Fetch class from module

X = Number(5)   # Number.__init__(X, 5)
Y = X - 2       # Number.__sub__(X, 2)
Y.data          # Y is new Number instance

# Common Operator-Overloading Methods
%run  -m timeit -n 10000 -r 10 -s "L = list(range(100))" "x = L.__len__()"
%run  -m timeit -n 10000 -r 10 -s "L = list(range(100))" "x = len(L)"

# Indexing and Slicing: __getitem__ and __setitem__
class Indexer:
    def __getitem__(self, index):
        return index ** 2
X = Indexer()
X[2]                        # X[i] calls X.__getitem__(i)

for i in range(5):
    print(X[i], end=' ')    # Runs __getitem__(X, i) each time

# Intercepting Slices
    # In addition to indexing, __getitem__ is also called for slice expressions
L = [5, 6, 7, 8, 9]
L[2:4]              # Slice with slice syntax: 2..(4-1)
L[1:]
L[:-1]
L[::2]

L[slice(2, 4)]      # Slice with slice objects
L[slice(1, None)]
L[slice(None, -1)]
L[slice(None, None, 2)]

class Indexer:
    def __init__(self, data):
        self.data = data
    def __getitem__(self, index):   # Called for index or slice
        print('getitem:', index)
        return self.data[index]     # Perform index or slice

X = Indexer([5, 6, 7, 8, 9])
X[0]                                # Indexing sends __getitem__ an integer
X[1]
X[-1]
X[2:4]                              # Slicing sends __getitem__ a slice object
X[1:]
X[:-1]
X[::2]

class Indexer:
    def __getitem__(self, index):
        if isinstance(index, int): # Test usage mode
            print('indexing', index)
        else:
            print('slicing', index.start, index.stop, index.step)
X = Indexer()
X[99]
X[1:99:2]
X[1:]

# Intercepting Item Assignments
# The __setitem__ index assignment method similarly intercepts both index and slice assignments
class IndexSetter:
    def __init__(self, data):
        self.data = data
    def __setitem__(self, index, value):    # Catch index or slice assignment
        print('setitem:', index)
        self.data[index] = value            # Assign index or slice

X = IndexSetter([5, 6, 7, 8, 9])
X[0] = 555
X[-2:] = [888, 999, 111]
X.data

# But __index__ Means As-Integer
# The __index__ method returns an integer value for an instance when one is needed.
class C:
    def __index__(self):
        return 255
X = C()
hex(X)      # Integer value
bin(X)
oct(X)

eds = [f'LP{i}e' for i in range(256)]
eds[255]
X = C()
eds[X]       # As index (not X[i]!)
eds[X:]      # As index (not X[i:]!)

# Index Iteration: __getitem__
class StepperIndex:
    def __getitem__(self, i):
        return self.data[i]
X = StepperIndex()          # X is a StepperIndex object
X.data = 'hack'
X[1]                        # Indexing calls __getitem__

for item in X:              # for loops call __getitem__
    print(item, end=' ')    # for indexes items 0..N

'k' in X                # All call __getitem__ too
[c for c in X]          # Comprehension
list(map(str.upper, X)) # map calls
(a, b, c, d) = X        # Sequence assignments
a, d
list(X), tuple(X), ''.join(X) # And so on...
X

# Iterable Objects: __iter__ and __next__
    # __iter__  supports general iteration tools better than  __getitem__ can

# User-Defined Iterables
from squares import Squares
for i in Squares(1, 5):     # for calls __iter__
    print(i, end=' ')       # Each iteration calls __next__

X = Squares(1, 5)       # Iterate manually: what loops do
I = iter(X)             # iter calls __iter__
next(I)                 # next calls __next__
next(I)
next(I)
next(I)                 # Can catch this in try statement
X = Squares(1, 5)
X[1]
list(X)[1]

# Single versus multiple scans
X = Squares(1, 5)           # Make an iterable with state
[n for n in X]              # Exhausts items: __iter__ returns self
[n for n in X]              # Now it's empty: __iter__ returns same self
[n for n in Squares(1, 5)]  # Make a new iterable object
list(Squares(1, 3))         # A new object for each new __iter__ call

36 in Squares(1, 10)        # Other iteration tools
a, b, c = Squares(1, 3)     # Each calls __iter__ and then __next__
a, b, c
':'.join(map(str, Squares(1, 5)))
X = Squares(1, 5)
tuple(X), tuple(X)          # Iterator exhausted in second tuple()

X = list(Squares(1, 5))
tuple(X), tuple(X)

# Classes versus generators
def gsquares(start, stop):              # Generator function
    for i in range(start, stop + 1):
        yield i ** 2

for i in gsquares(1, 5):
    print(i, end=' ')

for i in (x ** 2 for x in range(1, 6)): # Generator expression
    print(i, end=' ')

[x ** 2 for x in range(1, 6)]

# Multiple Iterators on One Object
S = 'ace'
for x in S:
    for y in S:
        print(x + y, end=' ')

# Multiple iterators with yield
%run skipper.py
# Classes versus slices
S = 'abcdef'
for x in S[::2]:
    for y in S[::2]:        # New objects on each iteration
        print(x + y, end=' ')

S = 'abcdef'
S = S[::2]

for x in S:
    for y in S:             # Same object, new iterators
        print(x + y, end=' ')

# Coding Alternative: __iter__ Plus yield
def gen(x):
    for i in range(x): yield i ** 2

G = gen(5)              # Create a generator with __iter__ and __next__
G.__iter__() is G       # Both methods exist on the same object
I = iter(G)             # Runs __iter__: generator returns itself
next(I), next(I)        # Runs __next__
list(gen(5))            # Iteration tools automatically run iter and next
##############
from squares_yield import Squares
for i in Squares(1, 5): print(i, end=' ')   # Runs __iter__, then __next__

S = Squares(1, 5)                           # Runs __init__: class saves instance state
I = iter(S)                                 # Runs __iter__: returns a generator
next(I)
next(I)                                     # Runs generator's __next__
next(I)                                     # Generator has both instance and local scope state
#############
from squares_yield_manual import Squares
for i in Squares(1, 5).gen(): print(i, end=' ')

S = Squares(1, 5)
I = iter(S.gen())                           # Call generator manually for iterable/iterator
next(I)

# Multiple iterators with yield
from squares_yield import Squares           # Using the __iter__/yield Squares

S = Squares(1, 5)
I = iter(S)
next(I); next(I)

K = iter(S)                                 # With yield, multiple iterators automatic
next(K)
next(I)                                     # I is independent of K: own local state
#####
S = Squares(1, 3)
for i in S:         # Each "for" calls __iter__
    for j in S:
        print(f'{i}:{j}', end=' ')

####
from squares_nonyield import Squares

for i in Squares(1, 5): print(i, end=' ')

S = Squares(1, 5)
I = iter(S)
next(I); next(I)
K = iter(S)         # Multiple iterators without yield
next(K)
next(I)
S = Squares(1, 3)
for i in S:         # Each "for" calls __iter__
    for j in S:
        print(f'{i}:{j}', end=' ')
####
from skipper_yield import SkipObject

skipper = SkipObject('abcdef')
I = iter(skipper)
next(I); next(I); next(I)
for x in skipper:           # Each "for" calls __iter__: new auto generator
    for y in skipper:
        print(x + y, end=' ')

# Membership: __contains__, __iter__, and __getitem__
%run contains.py

from contains import Iters

X = Iters('hack')
X[0]                # Indexing: __getitem__(0)
X[1:]               # Slicing: __getitem__(slice(…))
X[:-1]
list(X)
list(X)

# Attribute Access: __getattr__ and __setattr__
# Attribute Reference
class Empty:
    def __getattr__(self, attrname):    # On self.undefined
        if attrname == 'age':
            return 40
        else:
            raise AttributeError(attrname)
X = Empty()
X.age                                   # Becomes X.__getattr__('age')
X.name                                  # Unsupported attribute

# Attribute Assignment and Deletion
class Accesscontrol:
    def __setattr__(self, attr, value):
        if attr == 'age':
            self.__dict__[attr] = value + 10    # Not self.name=val or setattr
        else:
            raise AttributeError(attr + ' not allowed')

X = Accesscontrol()
X.age = 40                                      # Becomes X.__setattr__('age', 40)
X.age                                           # Found in __dict__ as usual
X.name = 'Pat'                                  # Unsupported attribute

####
self.age = value + 10                           # Loops!
setattr(self, attr, value + 10)                 # Loops! (attr is 'age')

self.other = 99                                 # Recurs + fails, but doesn't loop

object.__setattr__(self, attr, value + 10)      # OK: doesn't loop (preview)

# Other Attribute-Management Tools
    # __getattribute__ — catches every attribute fetch, so like __setattr__, any read inside it (e.g. self.x) would re-trigger itself → infinite loop. Fix: reroute through object.__getattribute__(self, attr) for the actual lookup.
    # property — lets you attach getter/setter functions to one specific named attribute. It's targeted, not generic — it can't intercept arbitrary/unknown attribute names, only the one you defined it for.
    # Descriptors — the lower-level protocol (__get__/__set__ on a class) that property is built on top of. Same idea as property (attached to one specific attribute), just more flexible/manual.
    # __slots__ — declared in the class to give named attributes fixed, implicit storage slots instead of a per-instance __dict__. Saves memory, but means there's no __dict__ to inspect/loop over — generic code (introspection, copying, etc.) must instead use __slots__-aware techniques (e.g. dir(), getattr/setattr, or iterating the slot names) to list/get/set attributes.

# Emulating Privacy for Instance Attributes: Part 1

# String Representation: __repr__ and __str__
class adder:
    def __init__(self, value=0):
        self.data = value       # Initialize data
    def __add__(self, other):
        self.data += other      # Add other in place
x = adder()                     # Default displays:
print(x)                        # str or else repr
x                               # repr

class addrepr(adder):                   # Inherit __init__, __add__
    def __repr__(self):                 # Add string representation
        return f'addrepr({self.data})'  # Convert to as-code string

x = addrepr(2)
x + 1
x               # Runs __repr__
addrepr(3)
print(x)        # Runs __repr__
addrepr(3)
str(x), repr(x) # Runs __repr__ for both

# Why Two Display Methods?
    # __str__ = nice for humans
    # __repr__ = useful for programmers

class addstr(adder):
    def __str__(self):                  # __str__ but no __repr__
        return f'[Value: {self.data}]'  # Convert to nice string
x = addstr(3)
x + 1
x                                       # Default __repr__ (in object)
print(x)                                # Runs __str__ (in addstr)
str(x), repr(x)

class addboth(adder):
    def __str__(self):
        return f'[Value: {self.data}]'  # User-friendly string
    def __repr__(self):
        return f'addboth({self.data})'  # As-code string
x = addboth(4)
x + 1
x                                       # Runs __repr__
addboth(5)
print(x)                                # Runs __str__
str(x), repr(x)

f'{x!s} {x!r}', '{!s} {!r}'.format(x, x), '%s %r' % (x, x)

# Display Usage Notes
class Printer:
    def __init__(self, val):
        self.val = val
    def __str__(self):              # Used for instance itself
        return str(self.val)        # Convert to a string result

objs = [Printer(2), Printer(3)]
for x in objs: print(x)             # __str__ run when instance printed
                                    # But not when instance is in a list!
print(objs)
objs

class Printer:
    def __init__(self, val):
        self.val = val
    def __repr__(self):             # __repr__ used by print if no __str__
        return str(self.val)        # __repr__ used if echoed or nested
objs = [Printer(2), Printer(3)]
for x in objs: print(x)             # No __str__: runs __repr__
print(objs)                         # Runs __repr__, not __str__
objs

# Right-Side and In-Place Ops: __radd__ and __iadd__
# Right-Side Addition
class Adder:
    def __init__(self, value=0):
        self.data = value
    def __add__(self, other):
        return self.data + other
x = Adder(5)
x + 2
2 + x

##
from commuter import Commuter1
x = Commuter1(88)
y = Commuter1(99)
x + 1
1 + y
x + y

# Reusing __add__ in __radd__
# Propagating class type
from commuter import Commuter5
x = Commuter5(88)
y = Commuter5(99)
x
x + 1                       # Result is another Commuter instance
1 + y
z = x + y                   # Not nested: doesn't recur to __radd__
z
z + 10
z + z
z + z + 1

z = x + y                   # With isinstance test+action commented-out
z
z + 10
z + z
z + z + 1
# In-Place Addition
class Number:
    def __init__(self, val):
        self.val = val
    def __iadd__(self, other):  # __iadd__ explicit: x += y
        self.val += other       # Usually returns self
        return self             # Else None is returned+assigned

x = Number(5)
x += 1
x += 1
x.val

y = Number([1])                 # In-place change faster than +
y += [2]
y += [3]
y.val

class Number:
    def __init__(self, val):
        self.val = val
    def __add__(self, other):           # __add__ fallback: x = (x + y)
        return Number(self.val + other) # Propagates class type

x = Number(5)
x += 1
x += 1                                  # And += does concatenation here
x.val

# Call Expressions: __call__
class Callee:
    def __call__(self, *pargs, **kargs):    # Intercept instance calls
        print(f'Called: {pargs=} {kargs=}') # Accept arbitrary arguments

C = Callee()
C(1, 2, 3)                                  # C is a "callable" object
C(1, 2, 3, x=4, y=5)

class C:
    def __call__(self, a, b, c=5, d=6): ...       # Normals and defaults
class C:
    def __call__(self, *pargs, **kargs): ...      # Collect arbitrary arguments
class C:
    def __call__(self, *pargs, d=6, **kargs): ... # 3.X keyword-only argument

X(1, 2)                             # Omit defaults
X(1, 2, 3, 4)                       # Positionals
X(a=1, b=2, d=4)                    # Keywords
X(*[1, 2], **dict(c=3, d=4))        # Unpack arbitrary arguments
X(1, *(2,), c=3, **dict(d=4))       # Mixed modes

class Prod:
    def __init__(self, value):      # Accept just one argument
        self.value = value
    def __call__(self, other):
        return self.value * other
x = Prod(2)                         # "Remembers" 2 in state
x(3)                                # 3 (passed) * 2 (state)
x(4)

class Prod:
    def __init__(self, value):
        self.value = value
    def comp(self, other):
        return self.value * other
x = Prod(3)
x.comp(3)

# Function Interfaces and Callback-Based Code
class Callback:
    def __init__(self, color):  # Function + state information
        self.color = color
    def __call__(self):         # Support calls with no arguments
        print('turn', self.color)

cb1 = Callback('blue')          # Remember blue
cb2 = Callback('green')         # Remember green
B1 = Button(command=cb1)        # Register handlers
B2 = Button(command=cb2)
cb1()                           # On event: prints 'turn blue'
cb2()                           # On event: prints 'turn green'

def callback(color):            # Enclosing scope versus attrs
    def oncall():
        print('turn', color)
    return oncall
cb3 = callback('yellow')        # Handler to be registered
cb3()                           # On event: prints 'turn yellow'

cb4 = (lambda color='red':      # Defaults retain state too
        print('turn', color))   # lambda defers code till call
cb4()                           # On event: prints 'turn red'


class Callback:
    def __init__(self, color): # Class with state information
        self.color = color
    def changeColor(self):     # A normally named method
        print('turn', self.color)
cb1 = Callback('blue')
cb2 = Callback('yellow')
B1 = Button(command=cb1.changeColor) # Bound method: reference, not call
B2 = Button(command=cb2.changeColor) # Remembers instance + function pair

cb1 = Callback('blue')
cmd = cb1.changeColor                # Registered event handler
cmd()                                # On event: prints 'turn blue'

# Comparisons: __lt__, __gt__, and Others
class Vetter:
    data = 'hack'
    def __gt__(self, other):
        print(f'gt: {self=} {other=}')
        return self.data > other
    def __lt__(self, other):
        print(f'lt: {self=} {other=}')
        return self.data < other
X = Vetter()
X > 'code', X < 'code'
'code' < X, 'code' > X
X < X

#  __lt__ is also used for sorts
class Order:
    def __init__(self, data):
        self.data = data
    def __lt__(self, other):
        return self.data < other.data
    def __repr__(self):
        return f'Order({self.data})'
sorted(Order(i) for i in [3, 1, 4, 2])

# Boolean Tests: __bool__ and __len__
class Truth:
    def __bool__(self): return True
X = Truth()
if X: print('yes!')
class Truth:
    def __bool__(self): return False
X = Truth()
bool(X)
##
class Truth:
    def __len__(self): return 0     # Empty means false too
X = Truth()
if not X: print('no!')
##
class Truth:
    def __bool__(self): return True # Preferred over length
    def __len__(self): return 0     # Object length: fallback

if Truth(): print('yes!')
##
class Truth:
    pass
X = Truth()
bool(X)

# Object Destruction: __del__
class Life:
    def __init__(self, name):
        print('Hello', name)
        self.name = name
    def live(self):
        print(self.name + '...')
    def __del__(self):
        print('Goodbye', self.name)
pat = Life('Pat')
pat.live()
pat = 'end'

# Destructor Usage Notes
    # __del__ is not very reliable: Python automatically manages memory, and you cannot always predict exactly when __del__ will run.
    # It can cause problems with exceptions, circular references, and program shutdown.
    # Therefore, prefer explicit cleanup such as close(), shutdown(), try/finally, or with instead of relying on __del__.